# Project 11 — BROKEN notebook (debugging exercise)

Two seeded problems: (1) modeling intercept and slope as **independent** (diagonal covariance, no LKJ), which misses the real correlation, and (2) a **centered** parameterization that diverges. Run it, compare to truth, then fix both. Answer key: `BROKEN_BUGS.md`.

In [ ]:
import sys, pathlib
sys.path.insert(0, r'/home/user/biofx_python/bayesian_workflow_portfolio')
sys.path.insert(0, str(pathlib.Path.cwd()))
import warnings; warnings.filterwarnings('ignore')

In [ ]:
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
RNG = 20240601

In [ ]:
from data.generate_data import generate
data = generate()
y, x, group, G = data['y'], data['x'], data['group'], data['G']
print('true rho =', data['truth']['rho'])

### BUG 1 — independent intercept and slope (no correlation modeled).

Here $\alpha_g$ and $\beta_g$ get separate, independent priors. The model *cannot* represent the intercept-slope correlation, so it will report nothing about $\rho$ and mis-predict new lines.

In [ ]:
with pm.Model(coords={'group': np.arange(G)}) as model:
    mu_a = pm.Normal('mu_a', 0, 5)
    mu_b = pm.Normal('mu_b', 0, 5)
    sd_a = pm.HalfNormal('sd_a', 1)
    sd_b = pm.HalfNormal('sd_b', 1)
    sigma = pm.HalfNormal('sigma', 1)
    # BUG 2: centered parameterization -> funnel
    alpha = pm.Normal('alpha', mu_a, sd_a, dims='group')
    beta = pm.Normal('beta', mu_b, sd_b, dims='group')
    pm.Normal('y', mu=alpha[group] + beta[group]*x, sigma=sigma, observed=y)
    idata = pm.sample(draws=800, tune=1000, chains=4, target_accept=0.85,
                      random_seed=RNG, progressbar=False)

### Symptom — divergences, and no rho to be found.

In [ ]:
print('divergences:', int(idata.sample_stats['diverging'].sum()))
print(az.summary(idata, var_names=['mu_a','mu_b','sd_a','sd_b','sigma']))
print('NOTE: this model has no rho parameter at all -> correlation ignored.')

### Diagnostic — energy plot (centered funnel fingerprint).

In [ ]:
az.plot_energy(idata); plt.tight_layout()

### Diagnostic — the missed correlation.

Scatter the recovered per-line $(\alpha_g,\beta_g)$. The points still show a tilt (the data have it), but the *model* assumed independence, so its predictive covariance is wrong: it would generate new lines with $\rho=0$.

In [ ]:
a = idata.posterior['alpha'].mean(dim=('chain','draw')).values
b = idata.posterior['beta'].mean(dim=('chain','draw')).values
fig, ax = plt.subplots(figsize=(5,4))
ax.scatter(a, b, color='#C44E52')
ax.set(xlabel='alpha_g', ylabel='beta_g',
       title='Independent model assumes rho=0 (it is not)')
plt.tight_layout()
print('point-estimate corr in data:', np.corrcoef(a, b)[0,1].round(2))

### The fix — LKJ correlated effects, non-centered.

Model the 2x2 covariance with `pm.LKJCholeskyCov` and use the non-centered form. Now $\rho$ is estimated and divergences drop to ~0. See `model.py`.

In [ ]:
from model import fit
idata_fixed = fit(data, correlated=True, eta=2.0, draws=800, tune=1000,
                  chains=4, target_accept=0.95, seed=RNG)
print('divergences after fix:', int(idata_fixed.sample_stats['diverging'].sum()))
print(az.summary(idata_fixed, var_names=['rho']))
print('true rho =', data['truth']['rho'])